In [137]:
import pandas as pd
import os

In [15]:
train_df=pd.read_parquet('train.parquet')
val_df=pd.read_parquet('val.parquet')
test_df=pd.read_parquet('test.parquet')

In [151]:
os.listdir('nyc-predict')

['test.csv',
 'GCP-Coupons-Instructions.rtf',
 'train.csv',
 'sample_submission.csv']

In [207]:
sub_df=pd.read_csv('./nyc-predict/sample_submission.csv')

In [17]:
train_df.sample(2)

,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,Distance,Day,Month,year,dayofweek,hour,jfk drop_dist,lga drop_dist,ewr drop_dist,met drop_dist,wtc drop_dist
216480,30.9,2012-04-22 00:23:00+00:00,-74.007715,40.725640,-73.961352,40.770173,1,6.31,22,4,2012,6,0,21.07,7.39,20.08,1.04,7.60
522733,8.5,2014-09-18 19:32:00+00:00,-73.990980,40.750547,-73.998220,40.730187,6,2.34,18,9,2014,3,19,21.03,11.68,15.53,6.22,2.19


In [19]:
train_df.columns

Index(['fare_amount', 'pickup_datetime', 'pickup_longitude', 'pickup_latitude',
       'dropoff_longitude', 'dropoff_latitude', 'passenger_count', 'Distance',
       'Day', 'Month', 'year', 'dayofweek', 'hour', 'jfk drop_dist',
       'lga drop_dist', 'ewr drop_dist', 'met drop_dist', 'wtc drop_dist'],
      dtype='object')

In [35]:
input_cols=['pickup_longitude', 'pickup_latitude',
       'dropoff_longitude', 'dropoff_latitude', 'passenger_count', 'Distance',
       'Day', 'Month', 'year', 'dayofweek', 'hour', 'jfk drop_dist',
       'lga drop_dist', 'ewr drop_dist', 'met drop_dist', 'wtc drop_dist']

In [37]:
target_cols='fare_amount'

In [41]:
train_inputs=train_df[input_cols]
train_targets=train_df[target_cols]

val_inputs=val_df[input_cols]
val_targets=val_df[target_cols]

test_inputs=test_df[input_cols]

In [199]:
sub_df

array([10.24160985, 10.24160985,  5.73178935, ..., 54.90753115,
       22.3193949 ,  7.26689813])

In [50]:
#models

In [52]:
#our basemodel had rmse of 9.6
#our best was decision tree- 3.97, 4.14 ((max_features=0.7,max_depth=10)

In [94]:
from sklearn.metrics import root_mean_squared_error as rmse

In [96]:
def test_model(model):
    model.fit(train_inputs,train_targets)
    t_pred=model.predict(train_inputs)
    train_rmse=rmse(train_targets,t_pred)
    v_pred=model.predict(val_inputs)
    val_rmse=rmse(val_targets,v_pred)
    res='train rmse: ',train_rmse,'\tval rmse: ',val_rmse
    return res

In [98]:
#1.ridge regressor
from sklearn.linear_model import Ridge

In [120]:
model=Ridge(alpha=0.8,random_state=42)

In [124]:
test_model(model)
#we see that it is better than our base model

('train rmse: ', 5.291232340062525, '\tval rmse: ', 5.150584780406425)

In [163]:
sub_df['fare_amount']=model.predict(test_inputs)

In [171]:
#sub_df.to_csv('ridge_sub.csv',index=False)
#submitting model - we got rmse score of 5.11

In [133]:
#2.decision tree
from sklearn.tree import DecisionTreeRegressor

In [221]:
model=DecisionTreeRegressor(random_state=42,max_depth=9)

In [223]:
test_model(model)

('train rmse: ', 3.9756187309295714, '\tval rmse: ', 4.19895012592601)

In [229]:
#lets try submitting this
"""
sub_df['fare_amount']=model.predict(test_inputs)
sub_df.to_csv('decision_tree_sub.csv',index=False)"""
#we have got rmse about 3.5 which is really good than previous model

"\nsub_df['fare_amount']=model.predict(test_inputs)\nsub_df.to_csv('decision_tree_sub.csv',index=False)"

In [232]:
#3. randomforest
from sklearn.ensemble import RandomForestRegressor

In [238]:
modelr=RandomForestRegressor(random_state=42,n_jobs=-1)

In [240]:
test_model(modelr)

('train rmse: ', 1.510640310585688, '\tval rmse: ', 3.815177282015823)

In [242]:
modelr=RandomForestRegressor(random_state=42,n_jobs=-1,max_depth=10,max_features=0.8)

In [244]:
test_model(modelr)

('train rmse: ', 3.687218367142636, '\tval rmse: ', 3.925622084048264)

In [248]:
test_model(RandomForestRegressor(random_state=42,n_jobs=-1,max_depth=10))

('train rmse: ', 3.7079810760500154, '\tval rmse: ', 3.9523178388355116)

In [250]:
modelr=RandomForestRegressor(random_state=42,n_jobs=-1,max_depth=10,max_features=0.8)

In [256]:
modelr.fit(train_inputs,train_targets)

RandomForestRegressor(max_depth=10, max_features=0.8, n_jobs=-1,
                      random_state=42)

In [258]:
modelr.predict(test_inputs)

array([10.33619778, 10.47263618,  5.07178917, ..., 54.42804238,
       22.08274521,  6.85122431])

In [266]:
"""sub_df['fare_amount']=modelr.predict(test_inputs)
sub_df.to_csv('random_forest_sub.csv',index=False)"""
#got about rmse = 3.37

"sub_df['fare_amount']=modelr.predict(test_inputs)\nsub_df.to_csv('random_forest_sub.csv',index=False)"

In [275]:
#4.xgboost
from xgboost import XGBRegressor as xg

In [277]:
modelx=xg(random_state=42,n_jobs=-1)

In [273]:
test_model(modelx)

('train rmse: ', 3.233098343498105, '\tval rmse: ', 3.805846552724725)

In [283]:
test_model(xg(random_state=42,n_jobs=-1,learning_rate=0.1,n_estimators=500))

('train rmse: ', 3.005378689693402, '\tval rmse: ', 3.7604845599874466)

In [291]:
test_model(xg(random_state=42,n_jobs=-1,learning_rate=0.09,n_estimators=500))

('train rmse: ', 3.047424335776834, '\tval rmse: ', 3.752054952617031)

In [342]:
modelx=xg(random_state=42,n_jobs=-1,learning_rate=0.09,n_estimators=500)

In [295]:
modelx.fit(train_inputs,train_targets)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.09, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=500, n_jobs=-1,
             num_parallel_tree=None, random_state=42, ...)

In [344]:
%%time
test_model(modelx)

CPU times: user 29.7 s, sys: 4.83 s, total: 34.5 s
Wall time: 4.52 s


('train rmse: ', 3.047424335776834, '\tval rmse: ', 3.752054952617031)

In [299]:
"""sub_df['fare_amount']=modelx.predict(test_inputs)
sub_df.to_csv('xgboost_sub.csv',index=False)"""
#got rmse of - 3.15
#we are in top 27% in leader board

"sub_df['fare_amount']=modelx.predict(test_inputs)\nsub_df.to_csv('xgboost_sub.csv',index=False)"

In [338]:
#we have got about 83 percentile with just 1 percent of training set and with very less time consuming model